# Automação de extração da base de Cargos
- Extrair do sistema Datamace a base de cargos através do módulo Cargos e Salários
- Tratamento da base CARGOS, padronização de nomes de colunas e formatos de dados
- Geração do arquivo em HTML
- Atualização do Power BI CARGOS E SALÁRIOS

# Importando as Bibliotecas

In [1]:
import duckdb as db
import logging
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import time
import tkinter as tk
import win32com.client
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from IPython.display import HTML, display
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
# Constantes de configuração

PATH_REGISTROS_PROCESSOS = Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx')
DIRETORIO_BUSCAR = Path(r'C:\Users\rodrigo.bernandes\Downloads')
DIRETORIO_MOVER = Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS')
CAMINHO_ARQUIVO_FINAL = Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS_DATAMACE.xlsx')
CAMINHO_CARGOS_SALARIOS = Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CARGOS E SALÁRIOS.xlsx')
CAMINHO_COLABORADORES = Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx')
ID_CARGA = 6  # ID para carga
ID_VALIDACAO = 19  # ID para validação

# Registra etapa do processo

In [3]:
def registrar_etapa(id_processo, etapa, wb_p=None, ws_p=None):
    """Registra uma etapa no controle de processos."""
    linha = [id_processo, datetime.today(), etapa]
    if ws_p is None:
        wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
        ws_p = wb_p['REGISTROS']
    ws_p.append(linha)
    wb_p.save(PATH_REGISTROS_PROCESSOS)
    logger.info(f"Etapa {etapa} registrada com sucesso.")
    return wb_p, ws_p

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais

In [4]:
automacao = r'X:\Gestao_de_Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\AUTOMACAO'

chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automacao}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-allow-origins=*")
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get("https://app.dtm.com.br/")
    wait = WebDriverWait(driver, 150)
    actions = ActionChains(driver)
    
    print("⏳ Sincronizando Datamace...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

⏳ Sincronizando Datamace...
🏁 Acesso finalizado


# Acessando o Datamace

In [5]:
pyautogui.PAUSE = 1

# Área de login na Cloud Datamace
pyautogui.click(x=748, y=381, clicks=2)
pyautogui.write("afpesp-17")
pyautogui.press("tab")
pyautogui.write("5SSWi3VqS")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(30) # Tempo maior para aguardar carregar a página de login

# Área de login do usuário
pyautogui.click(x=836, y=445)
pyautogui.write("RODRIGO")
pyautogui.press("tab")
pyautogui.write("Roddydio/")
pyautogui.press("tab")
pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Acesso ao Datamace realizado com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Acesso ao Datamace realizado com sucesso

----------------------------------------------------------------------------------------------------


# Dentro do Datamace, acessando o CS

In [6]:
# Acessando o CS
time.sleep(10) # Tempo maior para aguardar carregar o CS
pyautogui.click(x=273, y=195) # Clicando no módulo CS
time.sleep(2)
pyautogui.click(x=1079, y=235) # Fecha a janela de Tarefas Anuais
time.sleep(1)
pyautogui.click(x=449, y=297) # Cargos e Salários
time.sleep(1)
pyautogui.click(x=535, y=320) # Tabela Cargos/Funções
time.sleep(2)
pyautogui.click(x=715, y=323) # Tabela de Cargos
time.sleep(1)
pyautogui.click(x=907, y=723) # Imprimir
time.sleep(1)
pyautogui.click(x=749, y=602) # Saída
time.sleep(1)
pyautogui.click(x=733, y=683) # Excel
pyautogui.press("tab")
pyautogui.press("enter")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da base CARGOS

In [7]:
root = tk.Tk()
root.withdraw()  # não mostra janela principal

while True:
    tecla = simpledialog.askstring("Continuar", "Digite S para continuar:")
    if (tecla or "").strip().upper() == "S":
        break

print(" ✅ Continuando o processo...")

 ✅ Continuando o processo...


# Processo de carga e geração do arquivo em HTML

In [8]:
def processo_carga():
    """Processo de carga de cargos do Datamace."""
    logger.info("Iniciando processo de carga.")
    
    if not PATH_REGISTROS_PROCESSOS.exists():
        raise FileNotFoundError("Arquivo de registros de processos não encontrado.")
    
    wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
    ws_p = wb_p['REGISTROS']
    
    # Etapa 0: Início
    wb_p, ws_p = registrar_etapa(ID_CARGA, 0, wb_p, ws_p)
    
    # Verificar diretórios
    if not DIRETORIO_BUSCAR.exists():
        raise FileNotFoundError(f"Diretório de busca não encontrado: {DIRETORIO_BUSCAR}")
    if not DIRETORIO_MOVER.exists():
        DIRETORIO_MOVER.mkdir(parents=True, exist_ok=True)
    
    # Listar e filtrar arquivos GPS
    arquivos_gps = [f for f in DIRETORIO_BUSCAR.glob("*.XLSX") if f.name.startswith('GPS')]
    if not arquivos_gps:
        logger.error("Nenhum arquivo GPS encontrado no diretório.")
        wb_p, ws_p = registrar_etapa(ID_CARGA, 99, wb_p, ws_p)  # Etapa de erro
        raise FileNotFoundError("Nenhum arquivo GPS encontrado no diretório.")
    
    logger.info(f"Encontrados {len(arquivos_gps)} arquivos GPS.")
    
    # Etapa 1: Preparação
    wb_p, ws_p = registrar_etapa(ID_CARGA, 1, wb_p, ws_p)
    
    data_hoje = datetime.today().strftime('%Y-%m-%d')
    for arquivo in arquivos_gps:
        logger.info(f"Processando arquivo: {arquivo.name}")
        try:
            # Renomear para incluir data
            nome_arquivo_com_data = f"GPS_{data_hoje}.XLSX"
            arquivo_com_data = DIRETORIO_BUSCAR / nome_arquivo_com_data
            arquivo.rename(arquivo_com_data)
            
            # Carregar e tratar dados
            cargos = pd.read_excel(arquivo_com_data)
            
            # Verificar colunas necessárias (com acentos)
            colunas_esperadas = ['NÚMERO', 'CARGO', 'CG_DES_COM', 'CBO', 'STATUS', 'LEI APREND']
            colunas_faltantes = [c for c in colunas_esperadas if c not in cargos.columns]
            if colunas_faltantes:
                raise ValueError(f"Colunas faltantes no arquivo: {colunas_faltantes}")
            
            # Query DuckDB com aspas duplas para acentos
            cargos_tratado = db.query("""
                SELECT
                    "NÚMERO" AS id_Cargo,
                    "CARGO" AS cargo_abreviado,
                    "CG_DES_COM" AS cargo_completo,
                    "CBO" AS cbo,
                    "STATUS" AS status,
                    "LEI APREND" AS lei_aprendiz
                FROM cargos
            """).df()
            
            # Criar Excel com tabela formatada
            wb = Workbook()
            ws = wb.active
            ws.title = "CARGOS"
            
            for r in dataframe_to_rows(cargos_tratado, index=False, header=True):
                ws.append(r)
            
            # Adicionar tabela
            tabela_cargos = Table(displayName="CARGOSS", ref=ws.dimensions)
            estilo_tabela = TableStyleInfo(
                name="TableStyleLight13",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=True
            )
            tabela_cargos.tableStyleInfo = estilo_tabela
            ws.add_table(tabela_cargos)
            
            wb.save(CAMINHO_ARQUIVO_FINAL)
            logger.info(f"Arquivo salvo: {CAMINHO_ARQUIVO_FINAL}")
            
            # Mover arquivo processado
            destino = DIRETORIO_MOVER / nome_arquivo_com_data
            shutil.move(arquivo_com_data, destino)
            logger.info(f"Arquivo movido para: {destino}")
            
        except Exception as e:
            logger.error(f"Erro ao processar {arquivo.name}: {str(e)}")
            # Reverter renomeação se necessário (opcional)
            if 'arquivo_com_data' in locals() and arquivo_com_data.exists():
                arquivo_com_data.rename(arquivo)
    
    # Etapa 2: Finalização
    wb_p, ws_p = registrar_etapa(ID_CARGA, 2, wb_p, ws_p)
    logger.info("Processo de carga finalizado com sucesso.")

def limpar_coluna_numerica(df, coluna):
    """Limpa e converte coluna para inteiro, removendo não-numéricos."""
    logger.info(f"Limpando coluna '{coluna}'.")
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
    linhas_originais = len(df)
    df = df.dropna(subset=[coluna])
    df[coluna] = df[coluna].astype(int)
    linhas_removidas = linhas_originais - len(df)
    logger.info(f"Removidas {linhas_removidas} linhas não-numéricas em '{coluna}'.")
    if linhas_removidas > 0:
        logger.warning(f"AVISO: {linhas_removidas} linhas removidas de '{coluna}'.")
    return df

def processo_validacao():
    """Processo de validação de cargos e salários."""
    logger.info("Iniciando processo de validação.")
    
    if not PATH_REGISTROS_PROCESSOS.exists():
        raise FileNotFoundError("Arquivo de registros de processos não encontrado.")
    
    wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
    ws_p = wb_p['REGISTROS']
    
    # Etapa 0: Início
    wb_p, ws_p = registrar_etapa(ID_VALIDACAO, 0, wb_p, ws_p)
    
    hoje = datetime.today().date()
    data_br = hoje.strftime('%d/%m/%Y')
    caminho_arquivo_html = Path(r'X:\Gestao_de_Pessoas\Analytics\11 - Validações\11.1 - Cargos e Salários\2026') / f"CARGOS E SALÁRIOS - {data_br.replace('/', '-')}.html"
    
    # Verificar arquivos de entrada
    arquivos_necessarios = [CAMINHO_ARQUIVO_FINAL, CAMINHO_CARGOS_SALARIOS, CAMINHO_COLABORADORES]
    for arq in arquivos_necessarios:
        if not arq.exists():
            raise FileNotFoundError(f"Arquivo necessário não encontrado: {arq}")
    
    # Carregar bases
    cargos_datamace = pd.read_excel(CAMINHO_ARQUIVO_FINAL)
    cargos_salarios = pd.read_excel(CAMINHO_CARGOS_SALARIOS, usecols='B:L', skiprows=3)
    colaboradores = pd.read_excel(CAMINHO_COLABORADORES)
    
    # Limpeza
    cargos_datamace = limpar_coluna_numerica(cargos_datamace, 'id_Cargo')
    cargos_salarios = limpar_coluna_numerica(cargos_salarios, 'ID Datamace')
    
    # Registrar no DuckDB
    db.sql("CREATE OR REPLACE TABLE cargos_datamace AS SELECT * FROM cargos_datamace")
    db.sql("CREATE OR REPLACE TABLE cargos_salarios AS SELECT * FROM cargos_salarios")
    db.sql("CREATE OR REPLACE TABLE colaboradores AS SELECT * FROM colaboradores")
    
    logger.info("Bases carregadas e limpas no DuckDB.")
    
    # Validações (queries corrigidas) - ADICIONADAS LISTAS DETALHADAS PARA TODAS AS VALIDAÇÕES
    # 1. Cargos não cadastrados no controle GP (após reenquadramento)
    df_nao_cad_controle = db.query("""
        SELECT 
            A.id_Cargo AS "ID Datamace", 
            A.cargo_completo AS "Cargo Completo (Datamace)"
        FROM cargos_datamace AS A
        LEFT JOIN cargos_salarios AS B ON A.id_Cargo = B."ID Datamace"
        WHERE B."ID Datamace" IS NULL AND A.id_Cargo >= 834
        ORDER BY A.id_Cargo
    """).df()
    nao_cad_controle = len(df_nao_cad_controle)  # Contagem atualizada para len(df)
    logger.info(f"Query executada: {nao_cad_controle} cargos não cadastrados no controle GP.")
    
    # 2. Cargos antigos ativos
    df_antigos_ativos = db.query("""
        SELECT id_Cargo AS "ID Datamace", cargo_completo AS "Cargo Completo"
        FROM cargos_datamace
        WHERE id_Cargo < 834 AND status = 'Ativo'
        ORDER BY id_Cargo
    """).df()
    antigos_ativos = len(df_antigos_ativos)
    logger.info(f"Query executada: {antigos_ativos} cargos antigos ativos.")
    
    # 3. Descrição completa alterada
    df_descricao_completa = db.query("""
        SELECT 
            A.id_Cargo AS "ID Datamace",
            A.cargo_completo AS "Descrição Completa (Datamace)",
            B."Descrição Completa" AS "Descrição Completa (Tabela GP)"
        FROM cargos_datamace AS A
        LEFT JOIN cargos_salarios AS B ON A.id_Cargo = B."ID Datamace"
        WHERE B."ID Datamace" IS NOT NULL AND A.id_Cargo >= 834
          AND A.cargo_completo <> B."Descrição Completa"
        ORDER BY A.id_Cargo
    """).df()
    descricao_completa = len(df_descricao_completa)
    logger.info(f"Query executada: {descricao_completa} cargos com descrição completa alterada.")
    
    # 4. Descrição resumida alterada
    df_descricao_resumida = db.query("""
        SELECT 
            A.id_Cargo AS "ID Datamace",
            A.cargo_abreviado AS "Descrição Resumida (Datamace)",
            B."Descrição Resumida" AS "Descrição Resumida (Tabela GP)"
        FROM cargos_datamace AS A
        LEFT JOIN cargos_salarios AS B ON A.id_Cargo = B."ID Datamace"
        WHERE B."ID Datamace" IS NOT NULL AND A.id_Cargo >= 834
          AND A.cargo_abreviado <> B."Descrição Resumida"
        ORDER BY A.id_Cargo
    """).df()
    descricao_resumida = len(df_descricao_resumida)
    logger.info(f"Query executada: {descricao_resumida} cargos com descrição resumida alterada.")
    
    # 5. Divergências salariais (já com as colunas solicitadas)
    df_salario_dif = db.query("""
        SELECT
            A.registro AS "Matrícula",
            A.nome AS "Nome do colaborador",
            A.cargo_completo AS "Cargo",
            A.centro_custo AS "Centro de custo",
            A.salario_atual AS "Salário praticado",
            B."Salário" AS "Salário da tabela",
            A.hora_base AS "Carga horária"
        FROM colaboradores AS A
        LEFT JOIN cargos_salarios AS B ON A.cargo_completo = B."Descrição Completa"
        WHERE A.salario_atual <> B."Salário" AND A.descricao_rescisao = 'ATIVO'
        ORDER BY A.cargo_completo
    """).df()
    salario_dif = len(df_salario_dif)
    logger.info(f"Query executada: {salario_dif} colaboradores com salário divergente.")
    
    # CSS para tabelas (mantido o mesmo)
    css = """
    <style>
        .minha-tabela { border-collapse: collapse; width: 80%; margin: auto; }  /* Aumentei para 80% para caber mais colunas */
        .minha-tabela th, .minha-tabela td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        .minha-tabela th { background-color: #377AB4; color: white; }
        .minha-tabela tr:nth-child(even) { background-color: #f2f2f2; }
        .minha-tabela tr:hover { background-color: #ddd; }
    </style>
    """
    
    # Gerar HTML (adicionadas seções para as novas tabelas)
    html_titulo = f"""
    <div style="background-color: #23A638; padding: 1px; border: 3px solid #23A638;">
        <h1 style="color: #FFFFFF; font-size: 28px; font-family: 'Verdana'; font-weight: bold;"> VALIDAÇÃO DE CARGOS E SALÁRIOS - {data_br}</h1>
    </div>
    """
    
    # 1. Cargos não cadastrados
    html_nao_cad = df_nao_cad_controle.to_html(index=False, classes='minha-tabela', escape=False) if not df_nao_cad_controle.empty else "<p>Nenhum cargo não cadastrado encontrado.</p>"
    html_val_1 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos cadastrados após reenquadramento que não estão no controle GP: {nao_cad_controle}</p>
    </div>
    {html_nao_cad}
    """
    
    # 2. Cargos antigos ativos
    html_antigos = df_antigos_ativos.to_html(index=False, classes='minha-tabela', escape=False) if not df_antigos_ativos.empty else "<p>Nenhum cargo antigo ativo encontrado.</p>"
    html_val_2 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos antigos (anteriores ao reenquadramento) que ainda estão ativos: {antigos_ativos}</p>
    </div>
    {html_antigos}
    """
    
    # 3. Descrição completa alterada
    html_desc_completa = df_descricao_completa.to_html(index=False, classes='minha-tabela', escape=False) if not df_descricao_completa.empty else "<p>Nenhuma alteração na descrição completa encontrada.</p>"
    html_val_3 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição completa alterada: {descricao_completa}</p>
    </div>
    {html_desc_completa}
    """
    
    # 4. Descrição resumida alterada
    html_desc_resumida = df_descricao_resumida.to_html(index=False, classes='minha-tabela', escape=False) if not df_descricao_resumida.empty else "<p>Nenhuma alteração na descrição resumida encontrada.</p>"
    html_val_4 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de cargos que tiveram a descrição resumida alterada: {descricao_resumida}</p>
    </div>
    {html_desc_resumida}
    """
    
    # 5. Divergências salariais (colunas já solicitadas)
    html_salario_dif = df_salario_dif.to_html(index=False, classes='minha-tabela', escape=False) if not df_salario_dif.empty else "<p>Nenhuma divergência salarial encontrada.</p>"
    html_val_5 = f"""
    <div style="background-color: #FFFFFF; padding: 10px; border: 3px solid #23A638; height: 30px;">
        <p style="font-size: 18px; font-family: 'Verdana'; font-weight: bold;">Quantidade de colaboradores com o salário divergente da tabela: {salario_dif}</p>
    </div>
    {html_salario_dif}
    """
    
    html_resumo = css + '<br>' + html_titulo + '<br>' + html_val_1 + '<br>' + html_val_2 + '<br>' + html_val_3 + '<br>' + html_val_4 + '<br>' + html_val_5 + '<br>'
    
    # Exibir no Jupyter
    display(HTML(html_resumo))
    
    # Salvar HTML
    with open(caminho_arquivo_html, 'w', encoding='utf-8') as file:
        file.write(html_resumo)
    logger.info(f"Relatório HTML salvo em: {caminho_arquivo_html}")
    
    # Etapa 2: Finalização
    wb_p, ws_p = registrar_etapa(ID_VALIDACAO, 2, wb_p, ws_p)
    logger.info("Processo de validação finalizado com sucesso.")

def main():
    """Função principal: Executa carga seguida de validação."""
    logger.info("Iniciando fluxo completo: Carga + Validação.")
    try:
        processo_carga()
        processo_validacao()
        logger.info("Fluxo completo executado com sucesso.")
    except Exception as e:
        logger.error(f"Erro no fluxo principal: {str(e)}")
        # Registrar erro final no processo
        wb_p = load_workbook(PATH_REGISTROS_PROCESSOS)
        ws_p = wb_p['REGISTROS']
        registrar_etapa(ID_CARGA if 'carga' in str(e).lower() else ID_VALIDACAO, 99, wb_p, ws_p)
        raise

if __name__ == "__main__":
    main()

c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\rodrigo.bernandes\AppData\Local\Temp\ipykernel_18840\3739988354.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[coluna] = df[coluna].astype(int)
AVISO: 6 linhas removidas de 'ID Datamace'.


# Atualização do Power BI - CARGOS E SALÁRIOS

In [9]:
tempo_0 = [id, datetime.today(), 0]

pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\CARGOS E SALÁRIOS.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(20)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(40)
pyautogui.click(x=1294, y=109) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.click(x=712, y=379) # Clicar em GESTÃO DE PESSOAS
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(5)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI Atualizado

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [10]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:01:43.978314

----------------------------------------------------------------------------------------------------
